# 🔗 AI 밸류체인 고객→공급사 Lead-Lag Playground

고객(customer)이 공급사(supplier)로부터 구매하는 관계에서, **고객의 재무 활동이 공급사 매출을 몇 분기 선행하는가**를 스텝별로 분석합니다.

- **데이터:** 전처리 완료된 `panel_long` 테이블만 읽습니다 (파생변수 `cogs` 등은 분석 함수 내부에서 계산).
- **엔진:** `valuechain.py` (관계 정의 + 섹션 집계 + lead-lag) · `leadlag.py` (시차상관·Granger) · `panel.py` (데이터 접근).
- **가설:** `[고객] → [공급사]` 이면 고객이 선행(best lag > 0)한다.

아래 스텝을 순서대로 실행하세요. 각 스텝의 입력값을 바꿔가며 탐색할 수 있습니다.

## Step 1 — 데이터 로드 (`panel_long`)

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display
import panel as P
import valuechain as V

pd.set_option('display.max_rows', 120, 'display.width', 200)
long = P.load_long(P.DB_DEFAULT)          # 전처리된 롱테이블 전체 로드
print('panel_long:', long.shape, '| 회사수:', long.ticker.nunique(), '| 항목:', long.item.nunique())
display(long.head(4))

## Step 2 — 섹션 & 관계 정의 확인

`valuechain.py`에 정의된 고객→공급사 관계와, 각 섹션 집계에 쓰이는 회사를 확인합니다. Foundry·Server ODM은 이 DB에서 12분기 미만이라 신호를 만들 수 없습니다.

In [ ]:
print(f'고객→공급사 관계 {len(V.RELATIONSHIPS)}개:')
for c, s in V.RELATIONSHIPS:
    print(f'  {V.SECTIONS[c]:>18}  →  {V.SECTIONS[s]}')

print('\n섹션 구성 회사 (revenue 기준, 12분기+):')
rows = [{'섹션': lab, '회사': ', '.join(V.section_members(long, slug)) or '— 데이터 부족'}
        for slug, lab in V.SECTIONS.items()]
display(pd.DataFrame(rows))

## Step 3 — 섹션 집계 신호 만들기

한 섹션의 구성사 매출을 합산한 뒤 YoY→z-score로 변환한 신호를 봅니다. `SECTION`, `ITEM`을 바꿔 확인해 보세요.

In [ ]:
SECTION = 'ai_chip'      # hyperscalers, ai_chip, dram, nand, hw_equipment, ...
ITEM    = 'revenue'      # revenue, cogs, capex, inventory, ...

sig, raw, members = V.section_signal(long, SECTION, ITEM)
print(f'{V.SECTIONS[SECTION]} · {ITEM}  집계 회사:', members)
ax = sig.plot(marker='o', ms=3, figsize=(11,3), title=f'{V.SECTIONS[SECTION]} {ITEM} (YoY z-score)')
ax.axhline(0, color='grey', lw=.6); ax.set_xlabel('quarter'); display(ax.figure)

## Step 4 — 한 엣지 상세 분석 (핵심)

고객 섹션과 공급사 섹션, 그리고 각각의 재무변수(X=고객, Y=공급사)를 지정해 lead-lag를 계산합니다. `plot_edge`가 신호 오버레이 + 시차상관(CCF)을 그려줍니다.

In [ ]:
# ============== EDIT THESE ==============
CUSTOMER = 'ai_chip'     # 고객 섹션
SUPPLIER = 'dram'        # 공급사 섹션
X_ITEM   = 'revenue'     # 고객 변수 (revenue, cogs, capex, ...)
Y_ITEM   = 'revenue'     # 공급사 변수
# =======================================

edge = V.analyze_edge(long, CUSTOMER, SUPPLIER, X_ITEM, Y_ITEM)
display(pd.Series(edge)[['customer_label','supplier_label','status','best_lag_q','leader',
                          'corr','p_value','granger_c2s_p','granger_s2c_p','beta','r2','n','verdict']])
V.plot_edge(long, CUSTOMER, SUPPLIER, X_ITEM, Y_ITEM);

## Step 5 — 전체 매트릭스 (모든 관계 × 3종 재무관점)

23개 관계 × {매출→매출, 매출원가→매출, capex→매출} = 69 케이스를 한 번에 계산합니다.

In [ ]:
df = V.run_matrix(long)              # 기본 3종 관점
print('전체:', len(df), '| 분석가능:', (df.status=='ok').sum(),
      '| 고객선행&유의:', ((df.status=='ok') & (df.best_lag_q>0) & df.significant).sum())

# 매출→매출 관점만 표로
view0 = df[df.view.str.startswith('매출→매출')]
display(view0[['customer_label','supplier_label','status','best_lag_q','corr','p_value',
               'granger_c2s_p','verdict']])

## Step 6 — 관점별 요약 피벗 (best lag 분기)

In [ ]:
ok = df[df.status=='ok'].copy()
ok['edge'] = ok.customer_label + ' → ' + ok.supplier_label
pivot = ok.pivot_table(index='edge', columns='view', values='best_lag_q', aggfunc='first')
display(pivot)
print('양수 = 고객 선행 분기수 · NaN = 해당 관점 신호 없음/데이터 부족')

## Step 7 — 리포트(MD) + 밸류체인(HTML) 생성

결과를 `reports/`의 마크다운 리포트와, mermaid 밸류체인 다이어그램 HTML로 저장합니다.

In [ ]:
import os
os.makedirs('reports', exist_ok=True)
md   = V.generate_md(df, long, 'reports/valuechain_leadlag_2026-07-06.md')
html = V.generate_html(df, 'valuechain.html')            # 기본: 매출→매출 관점
print('저장됨:', md, '|', html)
# 다른 관점으로 HTML을 그리려면: V.generate_html(df, 'valuechain_cogs.html', view=df.view.unique()[1])